In [1]:
import cv2
cap = cv2.VideoCapture(0)
if not cap.isOpened(): raise SystemExit("No webcam found. Try index 1 or 2.")
while True:
    ok, frame = cap.read()
    if not ok: break
    cv2.imshow("Webcam", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break
cap.release(); cv2.destroyAllWindows()

KeyboardInterrupt: 

In [3]:
import cv2, time
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_draw  = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1,
                       min_detection_confidence=0.6, min_tracking_confidence=0.6)

prev = time.time()
while True:
    ok, frame = cap.read()
    if not ok: break
    frame = cv2.flip(frame, 1)  # mirror for natural control
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = hands.process(rgb)

    # draw landmarks + print index fingertip y
    if res.multi_hand_landmarks:
        lm = res.multi_hand_landmarks[0]
        mp_draw.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS)
        tip = lm.landmark[8]  # index fingertip
        cv2.putText(frame, f"index_y: {tip.y:.3f}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

    # fps overlay (nice to have)
    now = time.time()
    fps = 1/(now - prev) if now != prev else 0
    prev = now
    cv2.putText(frame, f"FPS: {fps:.0f}", (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

    cv2.imshow("Hand landmarks (press q to quit)", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release(); cv2.destroyAllWindows()
hands.close()